In [15]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [16]:
fake = Faker()

# Set random seed for generating same data each time 
Faker.seed(42)
np.random.seed(42)
random.seed(42)

NUM_PRODUCTIONS = 15000
NUM_SALES = 7500  # allow multiple sales per production

In [17]:

# Sample values for realism
crop_types = ['Wheat', 'Corn', 'Barley', 'Soybean', 'Potato']
varieties = {'Wheat': ['Durum', 'Emmer'], 'Corn': ['Sweet', 'Dent'], 'Barley': ['Hulled', 'Hulless'],
             'Soybean': ['Yellow', 'Black'], 'Potato': ['Russet', 'Red']}

fertilizers = ['NPK 15-15-15', 'Urea', 'Compost']
pesticides = ['Glyphosate', 'Chlorpyrifos', 'Neem Oil']
irrigation_types = ['Drip', 'Sprinkler', 'Flood']

buyer_types = ['Retailer', 'Wholesaler', 'Exporter']
channel_types = ['Direct', 'Online', 'Cooperative']

regions = ['Bavaria', 'Brandenburg', 'Saxony', 'Hesse', 'Lower Saxony']


In [18]:

def generate_production_data():
    records = []
    for i in range(1, NUM_PRODUCTIONS + 1):
        crop = random.choice(crop_types)
        variety = random.choice(varieties[crop])
        planting_date = fake.date_between(start_date='-2y', end_date='-6m')
        harvest_date = planting_date + timedelta(days=random.randint(90, 160))
        yield_kg = random.uniform(1000, 10000)
        avg_yield = yield_kg / random.uniform(0.5, 2.5)  # yield per hectare
        expected_price = round(random.uniform(0.3, 1.5), 2)
        records.append({
            'production_id': i,
            'farm_name': fake.company(),
            'field_id': f'F-{random.randint(100, 999)}',
            'field_location': fake.city(),
            'crop_type': crop,
            'crop_variety': variety,
            'planting_date': planting_date,
            'harvest_date': harvest_date,
            'fertilizer_type': random.choice(fertilizers),
            'fertilizer_quantity_kg': round(random.uniform(50, 300), 1),
            'pesticide_type': random.choice(pesticides),
            'pesticide_quantity_ltr': round(random.uniform(5, 30), 1),
            'irrigation_type': random.choice(irrigation_types),
            'irrigation_quantity_ltr': round(random.uniform(500, 2000), 1),
            'total_yield_kg': round(yield_kg, 1),
            'avg_yield_per_hectare': round(avg_yield, 1),
            'soil_ph': round(random.uniform(5.5, 7.5), 2),
            'rainfall_mm': round(random.uniform(200, 800), 1),
            'avg_temperature_c': round(random.uniform(12, 26), 1),
            'labor_hours': round(random.uniform(50, 200), 1),
            'number_of_workers': random.randint(2, 10),
            'machinery_used': random.choice(['Tractor', 'Harvester', 'Plough']),
            'expected_market_price_per_kg': expected_price,
            'expected_total_revenue_eur': round(expected_price * yield_kg, 2)
        })
    return pd.DataFrame(records)

In [19]:
# now generating the data
production_df = generate_production_data()

In [20]:
def generate_sales_data(production_df):
    records = []
    for i in range(1, NUM_SALES + 1):
        prod = production_df.sample(1).iloc[0]
        crop = prod['crop_type']
        variety = prod['crop_variety']
        market_price = prod['expected_market_price_per_kg']
        actual_price = round(market_price * random.uniform(0.9, 1.1), 2)
        quantity = round(random.uniform(200, 2000), 1)
        revenue = round(quantity * actual_price, 2)
        discount = round(random.uniform(0, 50), 2)
        shipping = round(random.uniform(10, 100), 2)
        net = revenue - discount - shipping
        records.append({
            'sales_id': i,
            'production_id': prod['production_id'],
            'crop_type': crop,
            'crop_variety': variety,
            'buyer_name': fake.company(),
            'buyer_type': random.choice(buyer_types),
            'buyer_region': random.choice(regions),
            'channel_type': random.choice(channel_types),
            'transaction_date': fake.date_between(start_date=prod['harvest_date'], end_date='today'),
            'quantity_sold_kg': quantity,
            'unit_price_eur': actual_price,
            'total_revenue_eur': revenue,
            'discount_applied_eur': discount,
            'net_revenue_eur': round(net, 2),
            'shipping_cost_eur': shipping,
            'profit_margin_pct': round(random.uniform(5, 25), 2),
            'market_price_per_kg': market_price,
            'price_variance': round(actual_price - market_price, 2)
        })
    return pd.DataFrame(records)


In [21]:
production_df.to_csv('fact_production.csv', index=False)


In [23]:
print("Production Sample:")
production_df.head()

Production Sample:


,production_id,farm_name,field_id,field_location,crop_type,crop_variety,planting_date,harvest_date,fertilizer_type,fertilizer_quantity_kg,...,total_yield_kg,avg_yield_per_hectare,soil_ph,rainfall_mm,avg_temperature_c,labor_hours,number_of_workers,machinery_used,expected_market_price_per_kg,expected_total_revenue_eur
0,1,Johnson LLC,F-858,East Donald,Wheat,Durum,2024-09-09,2025-01-12,Compost,71.7,...,3204.0,4112.6,6.51,215.9,14.8,147.5,10,Harvester,0.42,1345.69
1,2,"Yang, Gardner and Garza",F-814,North Claytonbury,Corn,Dent,2023-07-20,2023-11-22,Urea,135.1,...,8284.9,16149.9,6.26,415.4,16.8,89.7,2,Plough,1.27,10521.79
2,3,"Johnson, Gonzalez and Santos",F-982,Robinsonshire,Soybean,Yellow,2024-05-28,2024-10-13,Urea,194.3,...,1709.2,1573.3,6.08,247.9,15.3,65.2,6,Harvester,1.05,1794.66
3,4,Blake and Sons,F-799,Petersonberg,Barley,Hulled,2023-12-04,2024-04-19,Compost,67.9,...,4197.4,2280.8,6.42,362.0,25.0,153.2,5,Plough,1.14,4785.08
4,5,Garcia-James,F-167,Melanieview,Barley,Hulled,2023-10-16,2024-02-12,NPK 15-15-15,278.3,...,8396.2,3979.1,6.50,730.8,21.0,71.4,4,Tractor,0.78,6549.05


In [25]:
production_df.sample(1).iloc[0]

production_id                                      14991
farm_name                       Thomas, Tran and Swanson
field_id                                           F-117
field_location                              Destinyhaven
crop_type                                         Barley
crop_variety                                     Hulless
planting_date                                 2023-06-14
harvest_date                                  2023-10-17
fertilizer_type                             NPK 15-15-15
fertilizer_quantity_kg                             254.6
pesticide_type                              Chlorpyrifos
pesticide_quantity_ltr                              16.9
irrigation_type                                     Drip
irrigation_quantity_ltr                           1286.4
total_yield_kg                                    2437.2
avg_yield_per_hectare                             1369.5
soil_ph                                             6.96
rainfall_mm                    

In [24]:
sales_df = generate_sales_data(production_df)


ValueError: empty range for randrange() (1752451200, 1746144001, -6307199)